In [22]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedKFold
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report
import optuna


c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
df = pd.read_csv('../Milestone 3 EDA/ibm_hi_small_trans_cleaned.csv')
df

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Amount Paid,Is Laundering,Amount_Received_USD,Amount_Paid_USD,...,Payment Currency_Yuan,Payment Format_ACH,Payment Format_Bitcoin,Payment Format_Cash,Payment Format_Cheque,Payment Format_Credit Card,Payment Format_Reinvestment,Payment Format_Wire,Account_Same,Bank_Same
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.340000,3697.340000,0,3697.340000,3697.340000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.010000,0.010000,0,0.010000,0.010000,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.570000,14675.570000,0,14675.570000,14675.570000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.970000,2806.970000,0,2806.970000,2806.970000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.970000,36682.970000,0,36682.970000,36682.970000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5078340,2022/09/10 23:57,54219,8148A6631,256398,8148A8711,0.154978,0.154978,0,3107.386389,3107.386389,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078341,2022/09/10 23:35,15,8148A8671,256398,8148A8711,0.108128,0.108128,0,2168.020464,2168.020464,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078342,2022/09/10 23:52,154365,8148A6771,256398,8148A8711,0.004988,0.004988,0,100.011894,100.011894,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078343,2022/09/10 23:46,256398,8148A6311,256398,8148A8711,0.038417,0.038417,0,770.280058,770.280058,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,1


In [24]:
df.columns

Index(['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1',
       'Amount Received', 'Amount Paid', 'Is Laundering',
       'Amount_Received_USD', 'Amount_Paid_USD',
       ...
       'Payment Currency_Yuan', 'Payment Format_ACH', 'Payment Format_Bitcoin',
       'Payment Format_Cash', 'Payment Format_Cheque',
       'Payment Format_Credit Card', 'Payment Format_Reinvestment',
       'Payment Format_Wire', 'Account_Same', 'Bank_Same'],
      dtype='object', length=109)

In [25]:
df.drop(columns=['Timestamp', 'From Bank', 'To Bank', 'Account', 'Account.1'], inplace=True)

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5078345 entries, 0 to 5078344
Columns: 104 entries, Amount Received to Bank_Same
dtypes: float64(101), int64(3)
memory usage: 3.9 GB


In [27]:
random_state = 42

## Baseline Model on Entire Dataset

In [28]:
X = df.drop(columns='Is Laundering')
y = df['Is Laundering']

In [29]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.2, 
                                                    stratify=y,  # Maintains class distribution
                                                    random_state=random_state
                                                    )

### Default settings on RF

In [ ]:
# Train using optimal parameters
model = RandomForestClassifier(
    random_state=random_state,
    n_jobs=-1,
)
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [31]:
y_pred = model.predict(X_test)

In [32]:
print(classification_report(y_true=y_test, y_pred=y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1014634
           1       0.13      0.08      0.10      1035

    accuracy                           1.00   1015669
   macro avg       0.57      0.54      0.55   1015669
weighted avg       1.00      1.00      1.00   1015669



In [33]:
baseline_model_report_df = classification_report(y_test, y_pred, output_dict=True)

### Class Weight Balanced

In [34]:
# Train using optimal parameters
model = RandomForestClassifier(
    class_weight='balanced',
    random_state=random_state,
    n_jobs=-1
)
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [35]:
y_pred = model.predict(X_test)

In [36]:
print(classification_report(y_true=y_test, y_pred=y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1014634
           1       0.05      0.08      0.06      1035

    accuracy                           1.00   1015669
   macro avg       0.53      0.54      0.53   1015669
weighted avg       1.00      1.00      1.00   1015669



In [37]:
baseline_balanced_model_report_df = classification_report(y_test, y_pred, output_dict=True)

## Undersampling due to Class Imbalance

In [38]:
fraud = df[df['Is Laundering'] == 1]
non_fraud = df[df['Is Laundering'] == 0]
len(fraud), len(non_fraud)

(5177, 5073168)

In [39]:
# Undersample majority class to a 1:10 ratio
non_fraud_downsampled = non_fraud.sample(n=len(fraud) * 10, random_state=random_state)
non_fraud_downsampled

,Amount Received,Amount Paid,Is Laundering,Amount_Received_USD,Amount_Paid_USD,US Dollar_Amount_Paid_Mean,US Dollar_Amount_Paid_Median,US Dollar_Amount_Received_Mean,US Dollar_Amount_Received_Median,Bitcoin_Amount_Paid_Mean,...,Payment Currency_Yuan,Payment Format_ACH,Payment Format_Bitcoin,Payment Format_Cash,Payment Format_Cheque,Payment Format_Credit Card,Payment Format_Reinvestment,Payment Format_Wire,Account_Same,Bank_Same
4563287,1439.79,1439.79,0,3.829841e+02,3.829841e+02,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0,0
4216770,33.09,33.09,0,3.309000e+01,3.309000e+01,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0,0
2992877,263.34,263.34,0,3.807896e+01,3.807896e+01,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
1499016,17154983.76,17154983.76,0,1.747064e+07,1.747064e+07,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0,0
510282,8499.22,8499.22,0,2.501320e+03,2.501320e+03,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3992741,5722.87,5722.87,0,5.722870e+03,5.722870e+03,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0,0
44465,82208.92,82208.92,0,8.220892e+04,8.220892e+04,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0,0
1173046,6837.09,6837.09,0,5.196872e+03,5.196872e+03,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0,0
4125810,3261.27,3261.27,0,3.763506e+03,3.763506e+03,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0


In [40]:
# Combine and shuffle
df_reduced = pd.concat([fraud, non_fraud_downsampled]).sample(frac=1).reset_index(drop=True)
df_reduced

,Amount Received,Amount Paid,Is Laundering,Amount_Received_USD,Amount_Paid_USD,US Dollar_Amount_Paid_Mean,US Dollar_Amount_Paid_Median,US Dollar_Amount_Received_Mean,US Dollar_Amount_Received_Median,Bitcoin_Amount_Paid_Mean,...,Payment Currency_Yuan,Payment Format_ACH,Payment Format_Bitcoin,Payment Format_Cash,Payment Format_Cheque,Payment Format_Credit Card,Payment Format_Reinvestment,Payment Format_Wire,Account_Same,Bank_Same
0,184531.22,184531.22,1,1310.171662,1310.171662,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
1,3935.63,3935.63,0,4008.045592,4008.045592,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0,0
2,13507.02,13507.02,1,13432.731390,13432.731390,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
3,346.45,346.45,0,92.155700,92.155700,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0,0
4,465.90,465.90,0,465.900000,465.900000,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56942,321.04,321.04,0,94.482072,94.482072,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0
56943,548.67,548.67,0,372.492063,372.492063,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0,0
56944,128746.41,128746.41,0,1609.330125,1609.330125,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0,0
56945,1157.02,1157.02,1,1150.656390,1150.656390,367153.780541,904.13,4.041979e+06,908.2,20.809275,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0


In [41]:
X = df_reduced.drop(columns='Is Laundering')
y = df_reduced['Is Laundering']

In [42]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.2, 
                                                    stratify=y,  # Maintains class distribution
                                                    random_state=random_state
                                                    )

In [43]:
# cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)
from sklearn.model_selection import StratifiedKFold


cv = StratifiedKFold(n_splits=5)

### RF With unscaled Data and recall as param tuning metric

In [44]:
def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 10, 200)
    max_depth = trial.suggest_int("max_depth", 2, 32, log=True)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    
    model = RandomForestClassifier(
        class_weight='balanced',
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train, 
        y=y_train, 
        cv=cv,
        scoring='recall',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

unscaled_df = study.trials_dataframe().sort_values("value", ascending=False)
top_5_unscaled_df = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_unscaled_df)

[I 2026-04-14 18:56:32,741] A new study created in memory with name: no-name-17fe4f62-987c-4b9c-a003-2a0baa777005
[I 2026-04-14 18:56:38,150] Trial 0 finished with value: 0.7875491687208266 and parameters: {'n_estimators': 99, 'max_depth': 20, 'min_samples_split': 4}. Best is trial 0 with value: 0.7875491687208266.
[I 2026-04-14 18:56:40,722] Trial 1 finished with value: 0.8107235887484485 and parameters: {'n_estimators': 166, 'max_depth': 2, 'min_samples_split': 9}. Best is trial 1 with value: 0.8107235887484485.
[I 2026-04-14 18:56:43,120] Trial 2 finished with value: 0.818208889121985 and parameters: {'n_estimators': 72, 'max_depth': 5, 'min_samples_split': 8}. Best is trial 2 with value: 0.818208889121985.
[I 2026-04-14 18:56:45,437] Trial 3 finished with value: 0.8107238801186458 and parameters: {'n_estimators': 103, 'max_depth': 3, 'min_samples_split': 5}. Best is trial 2 with value: 0.818208889121985.
[I 2026-04-14 18:56:46,726] Trial 4 finished with value: 0.8085511325559576 an

,number,value,datetime_start,datetime_complete,duration,params_max_depth,params_min_samples_split,params_n_estimators,state
40,40,0.852489,2026-04-14 18:58:13.881564,2026-04-14 18:58:17.233389,0 days 00:00:03.351825,8,9,184,COMPLETE
74,74,0.852249,2026-04-14 18:59:42.305495,2026-04-14 18:59:45.889757,0 days 00:00:03.584262,9,10,179,COMPLETE
76,76,0.852249,2026-04-14 18:59:49.571348,2026-04-14 18:59:53.095867,0 days 00:00:03.524519,9,10,178,COMPLETE
41,41,0.852248,2026-04-14 18:58:17.234391,2026-04-14 18:58:20.390099,0 days 00:00:03.155708,8,9,182,COMPLETE
83,83,0.852248,2026-04-14 19:00:13.918548,2026-04-14 19:00:17.376644,0 days 00:00:03.458096,8,10,195,COMPLETE


In [45]:
top_5_unscaled_df

,number,value,datetime_start,datetime_complete,duration,params_max_depth,params_min_samples_split,params_n_estimators,state
40,40,0.852489,2026-04-14 18:58:13.881564,2026-04-14 18:58:17.233389,0 days 00:00:03.351825,8,9,184,COMPLETE
74,74,0.852249,2026-04-14 18:59:42.305495,2026-04-14 18:59:45.889757,0 days 00:00:03.584262,9,10,179,COMPLETE
76,76,0.852249,2026-04-14 18:59:49.571348,2026-04-14 18:59:53.095867,0 days 00:00:03.524519,9,10,178,COMPLETE
41,41,0.852248,2026-04-14 18:58:17.234391,2026-04-14 18:58:20.390099,0 days 00:00:03.155708,8,9,182,COMPLETE
83,83,0.852248,2026-04-14 19:00:13.918548,2026-04-14 19:00:17.376644,0 days 00:00:03.458096,8,10,195,COMPLETE


In [61]:
unscaled_df

,number,value,datetime_start,datetime_complete,duration,params_max_depth,params_min_samples_split,params_n_estimators,state
40,40,0.852489,2026-04-14 18:58:13.881564,2026-04-14 18:58:17.233389,0 days 00:00:03.351825,8,9,184,COMPLETE
74,74,0.852249,2026-04-14 18:59:42.305495,2026-04-14 18:59:45.889757,0 days 00:00:03.584262,9,10,179,COMPLETE
76,76,0.852249,2026-04-14 18:59:49.571348,2026-04-14 18:59:53.095867,0 days 00:00:03.524519,9,10,178,COMPLETE
41,41,0.852248,2026-04-14 18:58:17.234391,2026-04-14 18:58:20.390099,0 days 00:00:03.155708,8,9,182,COMPLETE
83,83,0.852248,2026-04-14 19:00:13.918548,2026-04-14 19:00:17.376644,0 days 00:00:03.458096,8,10,195,COMPLETE
95,95,0.852248,2026-04-14 19:00:52.605823,2026-04-14 19:00:55.804059,0 days 00:00:03.198236,8,9,173,COMPLETE
70,70,0.852248,2026-04-14 18:59:28.175511,2026-04-14 18:59:31.369146,0 days 00:00:03.193635,8,10,175,COMPLETE
71,71,0.852248,2026-04-14 18:59:31.370149,2026-04-14 18:59:34.730109,0 days 00:00:03.359960,8,10,186,COMPLETE
91,91,0.852248,2026-04-14 19:00:42.509170,2026-04-14 19:00:45.877642,0 days 00:00:03.368472,8,9,181,COMPLETE
90,90,0.852248,2026-04-14 19:00:39.147655,2026-04-14 19:00:42.508169,0 days 00:00:03.360514,8,9,178,COMPLETE


In [46]:
# Train using optimal parameters
model = RandomForestClassifier(
    class_weight='balanced',
    n_estimators=top_5_unscaled_df.head(1)['params_n_estimators'].item(),
    max_depth=top_5_unscaled_df.head(1)['params_max_depth'].item(),
    min_samples_split=top_5_unscaled_df.head(1)['params_min_samples_split'].item(),
    random_state=random_state
)
model.fit(X_train, y_train)

,n_estimators,184
,criterion,'gini'
,max_depth,8
,min_samples_split,9
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [47]:
y_pred = model.predict(X_test)

In [48]:
print(classification_report(y_true=y_test, y_pred=y_pred))

              precision    recall  f1-score   support

           0       0.98      0.93      0.96     10355
           1       0.54      0.84      0.66      1035

    accuracy                           0.92     11390
   macro avg       0.76      0.88      0.81     11390
weighted avg       0.94      0.92      0.93     11390



In [49]:
recall_model_report_df = classification_report(y_test, y_pred, output_dict=True)

### No scaling, f1 as eval metric

In [50]:
# Parameter tuning
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score


def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 10, 200)
    max_depth = trial.suggest_int("max_depth", 2, 32, log=True)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    
    model = RandomForestClassifier(
        class_weight='balanced',
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=random_state
    )
    
    scores = cross_val_score(
        model, 
        X=X_train, 
        y=y_train, 
        cv=cv,
        scoring="f1",
        n_jobs=-1,
    )
    f1 = scores.mean()
    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

f1_df = study.trials_dataframe().sort_values("value", ascending=False)
top_5_f1_df = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_f1_df)

[I 2026-04-14 19:01:12,818] A new study created in memory with name: no-name-92d17761-1650-4401-aefe-4299b0d3a93e
[I 2026-04-14 19:01:14,366] Trial 0 finished with value: 0.6640110440501512 and parameters: {'n_estimators': 143, 'max_depth': 4, 'min_samples_split': 10}. Best is trial 0 with value: 0.6640110440501512.
[I 2026-04-14 19:01:15,058] Trial 1 finished with value: 0.6191672801548862 and parameters: {'n_estimators': 62, 'max_depth': 3, 'min_samples_split': 8}. Best is trial 0 with value: 0.6640110440501512.
[I 2026-04-14 19:01:16,351] Trial 2 finished with value: 0.6520957929123067 and parameters: {'n_estimators': 129, 'max_depth': 3, 'min_samples_split': 4}. Best is trial 0 with value: 0.6640110440501512.
[I 2026-04-14 19:01:17,525] Trial 3 finished with value: 0.6445321610810838 and parameters: {'n_estimators': 130, 'max_depth': 2, 'min_samples_split': 9}. Best is trial 0 with value: 0.6640110440501512.
[I 2026-04-14 19:01:19,918] Trial 4 finished with value: 0.669119196837343

,number,value,datetime_start,datetime_complete,duration,params_max_depth,params_min_samples_split,params_n_estimators,state
75,75,0.699392,2026-04-14 19:06:11.483062,2026-04-14 19:06:15.411720,0 days 00:00:03.928658,17,2,137,COMPLETE
51,51,0.699267,2026-04-14 19:04:24.947529,2026-04-14 19:04:29.401116,0 days 00:00:04.453587,17,2,146,COMPLETE
72,72,0.699052,2026-04-14 19:05:58.171563,2026-04-14 19:06:02.705274,0 days 00:00:04.533711,17,2,157,COMPLETE
53,53,0.698983,2026-04-14 19:04:34.310195,2026-04-14 19:04:38.971535,0 days 00:00:04.661340,17,2,150,COMPLETE
62,62,0.698804,2026-04-14 19:05:11.213238,2026-04-14 19:05:15.704903,0 days 00:00:04.491665,18,2,142,COMPLETE


In [60]:
f1_df

,number,value,datetime_start,datetime_complete,duration,params_max_depth,params_min_samples_split,params_n_estimators,state
75,75,0.699392,2026-04-14 19:06:11.483062,2026-04-14 19:06:15.411720,0 days 00:00:03.928658,17,2,137,COMPLETE
51,51,0.699267,2026-04-14 19:04:24.947529,2026-04-14 19:04:29.401116,0 days 00:00:04.453587,17,2,146,COMPLETE
72,72,0.699052,2026-04-14 19:05:58.171563,2026-04-14 19:06:02.705274,0 days 00:00:04.533711,17,2,157,COMPLETE
53,53,0.698983,2026-04-14 19:04:34.310195,2026-04-14 19:04:38.971535,0 days 00:00:04.661340,17,2,150,COMPLETE
62,62,0.698804,2026-04-14 19:05:11.213238,2026-04-14 19:05:15.704903,0 days 00:00:04.491665,18,2,142,COMPLETE
52,52,0.698613,2026-04-14 19:04:29.402113,2026-04-14 19:04:34.309194,0 days 00:00:04.907081,17,2,155,COMPLETE
71,71,0.698476,2026-04-14 19:05:53.314882,2026-04-14 19:05:58.169563,0 days 00:00:04.854681,17,2,162,COMPLETE
82,82,0.698281,2026-04-14 19:06:39.618725,2026-04-14 19:06:44.937835,0 days 00:00:05.319110,17,2,171,COMPLETE
65,65,0.697729,2026-04-14 19:05:24.513892,2026-04-14 19:05:29.868677,0 days 00:00:05.354785,18,2,173,COMPLETE
60,60,0.697436,2026-04-14 19:05:01.474925,2026-04-14 19:05:06.340623,0 days 00:00:04.865698,18,3,163,COMPLETE


In [51]:
# Train using optimal parameters
model = RandomForestClassifier(
    class_weight='balanced',
    n_estimators=top_5_f1_df.head(1)['params_n_estimators'].item(),
    max_depth=top_5_f1_df.head(1)['params_max_depth'].item(),
    min_samples_split=top_5_f1_df.head(1)['params_min_samples_split'].item(),
    random_state=random_state
)
model.fit(X_train, y_train)

,n_estimators,137
,criterion,'gini'
,max_depth,17
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [52]:
y_pred = model.predict(X_test)

In [53]:
print(classification_report(y_true=y_test, y_pred=y_pred))

              precision    recall  f1-score   support

           0       0.98      0.95      0.96     10355
           1       0.60      0.79      0.69      1035

    accuracy                           0.93     11390
   macro avg       0.79      0.87      0.82     11390
weighted avg       0.94      0.93      0.94     11390



In [54]:
f1_model_report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)

### Compare Models

In [62]:
# 2. Convert and transpose
recall_report_df = pd.DataFrame(recall_model_report_df).transpose()
f1_report_df = pd.DataFrame(f1_model_report).transpose()

# 3. Combine multiple reports (e.g., adding a 'Model' column for identification)
combined_df = pd.concat([recall_report_df, f1_report_df], keys=['IBM RF optimized for Recall', 'IBM RF optimized for F1'])

In [63]:
combined_df

precision    recall  f1-score  \
IBM RF optimized for Recall 0              0.983030  0.928634  0.955058   
                            1              0.540423  0.839614  0.657586   
                            accuracy       0.920544  0.920544  0.920544   
                            macro avg      0.761726  0.884124  0.806322   
                            weighted avg   0.942811  0.920544  0.928027   
IBM RF optimized for F1     0              0.978766  0.948141  0.963210   
                            1              0.604857  0.794203  0.686717   
                            accuracy       0.934153  0.934153  0.934153   
                            macro avg      0.791811  0.871172  0.824963   
                            weighted avg   0.944789  0.934153  0.938085   

                                               support  
IBM RF optimized for Recall 0             10355.000000  
                            1              1035.000000  
                            accuracy          0.920544  
                            macro avg     11390.000000  
                            weighted avg  11390.000000  
IBM RF optimized for F1     0             10355.000000  
                            1              1035.000000  
                            accuracy          0.934153  
                            macro avg     11390.000000  
                            weighted avg  11390.000000

In [57]:
model.feature_importances_

array([0.08335239, 0.08734736, 0.10743503, 0.11814191, 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.000488  ,
       0.00145135, 0.00052624, 0.00057799, 0.00206075, 0.00056088,
       0.00072504, 0.00059483, 0.00486041, 0.00144581, 0.00077

In [58]:
pd.set_option('display.max_rows', None)

In [59]:
importances = model.feature_importances_
feature_names = df.drop(columns='Is Laundering').columns
feature_imp_df = pd.DataFrame({'Feature': feature_names, 'Gini Importance': importances}).sort_values(
    'Gini Importance', ascending=False)
feature_imp_df

,Feature,Gini Importance
94,Payment Format_ACH,0.333700
3,Amount_Paid_USD,0.118142
2,Amount_Received_USD,0.107435
1,Amount Paid,0.087347
97,Payment Format_Cheque,0.087080
0,Amount Received,0.083352
98,Payment Format_Credit Card,0.060629
101,Account_Same,0.027424
96,Payment Format_Cash,0.019111
102,Bank_Same,0.016327
